# F1 Race Predictor — Phase 2: Feature Engineering

**Input:** `data/driver_race_results.csv`  
**Output:** `data/features.csv`

Run cells top to bottom. The penalty fetch cell (Section 9) takes ~5 min on first run; re-running is instant (cached).

In [42]:
import sys
print(sys.executable)

/Users/aksha/Documents/C/CoDeS/AI_Engineer/FastF1/venv/bin/python


In [43]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import requests
import time
import json
from pathlib import Path

In [44]:
df = pd.read_csv("data/driver_race_results.csv")
df["q1_time"] = pd.to_timedelta(df["q1_time"])
df["q2_time"] = pd.to_timedelta(df["q2_time"])
df["q3_time"] = pd.to_timedelta(df["q3_time"])
print(df.shape)
df.head()

(3612, 16)


,season,round,circuit_id,driver_id,driver_full_name,team_id,grid_position,finish_position,classified_status,points,q1_time,q2_time,q3_time,race_avg_air_temp,race_avg_track_temp,race_had_rain
0,2018,1,Australian Grand Prix,VET,Sebastian Vettel,Ferrari,3.0,1.0,Finished,25.0,0 days 00:01:23.348000,0 days 00:01:21.944000,0 days 00:01:21.838000,24.077477,36.324324,True
1,2018,1,Australian Grand Prix,HAM,Lewis Hamilton,Mercedes,1.0,2.0,Finished,18.0,0 days 00:01:22.824000,0 days 00:01:22.051000,0 days 00:01:21.164000,24.077477,36.324324,True
2,2018,1,Australian Grand Prix,RAI,Kimi Räikkönen,Ferrari,2.0,3.0,Finished,15.0,0 days 00:01:23.096000,0 days 00:01:22.507000,0 days 00:01:21.828000,24.077477,36.324324,True
3,2018,1,Australian Grand Prix,RIC,Daniel Ricciardo,Red Bull Racing,8.0,4.0,Finished,12.0,0 days 00:01:23.494000,0 days 00:01:22.897000,0 days 00:01:22.152000,24.077477,36.324324,True
4,2018,1,Australian Grand Prix,ALO,Fernando Alonso,McLaren,10.0,5.0,Finished,10.0,0 days 00:01:23.597000,0 days 00:01:23.692000,NaT,24.077477,36.324324,True


In [45]:
print(df.groupby("season")["round"].nunique())

season
2018    21
2019    21
2020    17
2021    22
2022    22
2023    22
2024    24
2025    24
2026     7
Name: round, dtype: int64


# 1. Overtaking Difficulty (var_delta_i_norm)

Variance of normalised grid-to-finish delta per race. High = easy to overtake. Low = Monaco-style processional.

In [46]:
# per driver per race: delta = finish - grid
df["delta_i"] = df["finish_position"] - df["grid_position"]

# drivers per race (for normalisation)
drivers_per_race = (
    df.groupby(["season","round","circuit_id"])["driver_id"]
      .count()
      .reset_index(name="drivers_count")
)
df2 = df.merge(drivers_per_race, on=["season","circuit_id","round"], how="left")
df2["delta_i_norm"] = df2["delta_i"] / df2["drivers_count"]
df2.head()

,season,round,circuit_id,driver_id,driver_full_name,team_id,grid_position,finish_position,classified_status,points,q1_time,q2_time,q3_time,race_avg_air_temp,race_avg_track_temp,race_had_rain,delta_i,drivers_count,delta_i_norm
0,2018,1,Australian Grand Prix,VET,Sebastian Vettel,Ferrari,3.0,1.0,Finished,25.0,0 days 00:01:23.348000,0 days 00:01:21.944000,0 days 00:01:21.838000,24.077477,36.324324,True,-2.0,20,-0.10
1,2018,1,Australian Grand Prix,HAM,Lewis Hamilton,Mercedes,1.0,2.0,Finished,18.0,0 days 00:01:22.824000,0 days 00:01:22.051000,0 days 00:01:21.164000,24.077477,36.324324,True,1.0,20,0.05
2,2018,1,Australian Grand Prix,RAI,Kimi Räikkönen,Ferrari,2.0,3.0,Finished,15.0,0 days 00:01:23.096000,0 days 00:01:22.507000,0 days 00:01:21.828000,24.077477,36.324324,True,1.0,20,0.05
3,2018,1,Australian Grand Prix,RIC,Daniel Ricciardo,Red Bull Racing,8.0,4.0,Finished,12.0,0 days 00:01:23.494000,0 days 00:01:22.897000,0 days 00:01:22.152000,24.077477,36.324324,True,-4.0,20,-0.20
4,2018,1,Australian Grand Prix,ALO,Fernando Alonso,McLaren,10.0,5.0,Finished,10.0,0 days 00:01:23.597000,0 days 00:01:23.692000,NaT,24.077477,36.324324,True,-5.0,20,-0.25


In [47]:
# variance per race → overtaking difficulty score
race_variance = (
    df2.groupby(["season","round","circuit_id"])["delta_i_norm"]
       .var()
       .reset_index(name="var_delta_i_norm")
)
race_variance.head()

,season,round,circuit_id,var_delta_i_norm
0,2018,1,Australian Grand Prix,0.051316
1,2018,2,Bahrain Grand Prix,0.113684
2,2018,3,Chinese Grand Prix,0.031579
3,2018,4,Azerbaijan Grand Prix,0.135526
4,2018,5,Spanish Grand Prix,0.071053


In [48]:
df = df.merge(race_variance, on=["season","round","circuit_id"], how="left")
print(f"var_delta_i_norm nulls: {df['var_delta_i_norm'].isna().sum()}")
df.head()

var_delta_i_norm nulls: 44


,season,round,circuit_id,driver_id,driver_full_name,team_id,grid_position,finish_position,classified_status,points,q1_time,q2_time,q3_time,race_avg_air_temp,race_avg_track_temp,race_had_rain,delta_i,var_delta_i_norm
0,2018,1,Australian Grand Prix,VET,Sebastian Vettel,Ferrari,3.0,1.0,Finished,25.0,0 days 00:01:23.348000,0 days 00:01:21.944000,0 days 00:01:21.838000,24.077477,36.324324,True,-2.0,0.051316
1,2018,1,Australian Grand Prix,HAM,Lewis Hamilton,Mercedes,1.0,2.0,Finished,18.0,0 days 00:01:22.824000,0 days 00:01:22.051000,0 days 00:01:21.164000,24.077477,36.324324,True,1.0,0.051316
2,2018,1,Australian Grand Prix,RAI,Kimi Räikkönen,Ferrari,2.0,3.0,Finished,15.0,0 days 00:01:23.096000,0 days 00:01:22.507000,0 days 00:01:21.828000,24.077477,36.324324,True,1.0,0.051316
3,2018,1,Australian Grand Prix,RIC,Daniel Ricciardo,Red Bull Racing,8.0,4.0,Finished,12.0,0 days 00:01:23.494000,0 days 00:01:22.897000,0 days 00:01:22.152000,24.077477,36.324324,True,-4.0,0.051316
4,2018,1,Australian Grand Prix,ALO,Fernando Alonso,McLaren,10.0,5.0,Finished,10.0,0 days 00:01:23.597000,0 days 00:01:23.692000,NaT,24.077477,36.324324,True,-5.0,0.051316


# 3. Rolling Team Finishing Position

Average team finish over last 3 races.

In [49]:
df_avg_finish = (
    df.groupby(["season","round","team_id"])["finish_position"]
      .mean()
      .reset_index(name="avg_team_finish_position")
      .sort_values(["team_id","season","round"])
)
df_avg_finish["rolling_avg_finish_position"] = (
    df_avg_finish.groupby("team_id")["avg_team_finish_position"]
                 .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)
df = df.merge(
    df_avg_finish[["season","round","team_id","avg_team_finish_position","rolling_avg_finish_position"]],
    on=["season","round","team_id"], how="left"
)
print(f"rolling_avg_finish_position nulls: {df['rolling_avg_finish_position'].isna().sum()}")
df.head()

rolling_avg_finish_position nulls: 0


,season,round,circuit_id,driver_id,driver_full_name,team_id,grid_position,finish_position,classified_status,points,q1_time,q2_time,q3_time,race_avg_air_temp,race_avg_track_temp,race_had_rain,delta_i,var_delta_i_norm,avg_team_finish_position,rolling_avg_finish_position
0,2018,1,Australian Grand Prix,VET,Sebastian Vettel,Ferrari,3.0,1.0,Finished,25.0,0 days 00:01:23.348000,0 days 00:01:21.944000,0 days 00:01:21.838000,24.077477,36.324324,True,-2.0,0.051316,2.0,2.0
1,2018,1,Australian Grand Prix,HAM,Lewis Hamilton,Mercedes,1.0,2.0,Finished,18.0,0 days 00:01:22.824000,0 days 00:01:22.051000,0 days 00:01:21.164000,24.077477,36.324324,True,1.0,0.051316,5.0,5.0
2,2018,1,Australian Grand Prix,RAI,Kimi Räikkönen,Ferrari,2.0,3.0,Finished,15.0,0 days 00:01:23.096000,0 days 00:01:22.507000,0 days 00:01:21.828000,24.077477,36.324324,True,1.0,0.051316,2.0,2.0
3,2018,1,Australian Grand Prix,RIC,Daniel Ricciardo,Red Bull Racing,8.0,4.0,Finished,12.0,0 days 00:01:23.494000,0 days 00:01:22.897000,0 days 00:01:22.152000,24.077477,36.324324,True,-4.0,0.051316,5.0,5.0
4,2018,1,Australian Grand Prix,ALO,Fernando Alonso,McLaren,10.0,5.0,Finished,10.0,0 days 00:01:23.597000,0 days 00:01:23.692000,NaT,24.077477,36.324324,True,-5.0,0.051316,7.0,7.0


# 4. Driver Circuit History

`career_starts_at_circuit` and `avg_finish_at_circuit` (historical average, excluding current race).

In [50]:
df_sorted = df.sort_values(["driver_id","circuit_id","season","round"])

df["career_starts_at_circuit"] = (
    df_sorted.groupby(["driver_id","circuit_id"]).cumcount()
)
df["avg_finish_at_circuit"] = (
    df_sorted.groupby(["driver_id","circuit_id"])["finish_position"]
             .transform(lambda x: x.shift(1).expanding().mean())
)
print(f"avg_finish_at_circuit nulls: {df['avg_finish_at_circuit'].isna().sum()} (expected: first appearance)")
df.head()

avg_finish_at_circuit nulls: 1202 (expected: first appearance)


,season,round,circuit_id,driver_id,driver_full_name,team_id,grid_position,finish_position,classified_status,points,...,q3_time,race_avg_air_temp,race_avg_track_temp,race_had_rain,delta_i,var_delta_i_norm,avg_team_finish_position,rolling_avg_finish_position,career_starts_at_circuit,avg_finish_at_circuit
0,2018,1,Australian Grand Prix,VET,Sebastian Vettel,Ferrari,3.0,1.0,Finished,25.0,...,0 days 00:01:21.838000,24.077477,36.324324,True,-2.0,0.051316,2.0,2.0,0,NaN
1,2018,1,Australian Grand Prix,HAM,Lewis Hamilton,Mercedes,1.0,2.0,Finished,18.0,...,0 days 00:01:21.164000,24.077477,36.324324,True,1.0,0.051316,5.0,5.0,0,NaN
2,2018,1,Australian Grand Prix,RAI,Kimi Räikkönen,Ferrari,2.0,3.0,Finished,15.0,...,0 days 00:01:21.828000,24.077477,36.324324,True,1.0,0.051316,2.0,2.0,0,NaN
3,2018,1,Australian Grand Prix,RIC,Daniel Ricciardo,Red Bull Racing,8.0,4.0,Finished,12.0,...,0 days 00:01:22.152000,24.077477,36.324324,True,-4.0,0.051316,5.0,5.0,0,NaN
4,2018,1,Australian Grand Prix,ALO,Fernando Alonso,McLaren,10.0,5.0,Finished,10.0,...,NaT,24.077477,36.324324,True,-5.0,0.051316,7.0,7.0,0,NaN


# 5. Quali-to-Race Delta

`finish_grid_delta` (per race) and `avg_finish_grid_delta` (rolling historical average).

In [51]:
df["finish_grid_delta"] = df["finish_position"] - df["grid_position"]

df_sorted2 = df.sort_values(["driver_id","season","round"])
df["avg_finish_grid_delta"] = (
    df_sorted2.groupby("driver_id")["finish_grid_delta"]
              .transform(lambda x: x.shift(1).expanding().mean())
)
print(f"avg_finish_grid_delta nulls: {df['avg_finish_grid_delta'].isna().sum()}")
df.head()

avg_finish_grid_delta nulls: 44


,season,round,circuit_id,driver_id,driver_full_name,team_id,grid_position,finish_position,classified_status,points,...,race_avg_track_temp,race_had_rain,delta_i,var_delta_i_norm,avg_team_finish_position,rolling_avg_finish_position,career_starts_at_circuit,avg_finish_at_circuit,finish_grid_delta,avg_finish_grid_delta
0,2018,1,Australian Grand Prix,VET,Sebastian Vettel,Ferrari,3.0,1.0,Finished,25.0,...,36.324324,True,-2.0,0.051316,2.0,2.0,0,NaN,-2.0,NaN
1,2018,1,Australian Grand Prix,HAM,Lewis Hamilton,Mercedes,1.0,2.0,Finished,18.0,...,36.324324,True,1.0,0.051316,5.0,5.0,0,NaN,1.0,NaN
2,2018,1,Australian Grand Prix,RAI,Kimi Räikkönen,Ferrari,2.0,3.0,Finished,15.0,...,36.324324,True,1.0,0.051316,2.0,2.0,0,NaN,1.0,NaN
3,2018,1,Australian Grand Prix,RIC,Daniel Ricciardo,Red Bull Racing,8.0,4.0,Finished,12.0,...,36.324324,True,-4.0,0.051316,5.0,5.0,0,NaN,-4.0,NaN
4,2018,1,Australian Grand Prix,ALO,Fernando Alonso,McLaren,10.0,5.0,Finished,10.0,...,36.324324,True,-5.0,0.051316,7.0,7.0,0,NaN,-5.0,NaN


# 6. Teammate Head-to-Head

`teammate_wins`, `teammate_losses`, `teammate_win_pct` — career aggregate.

In [52]:
def teammate_result(group):
    group = group.sort_values("finish_position")
    if len(group) < 2:
        return None
    winner = group.iloc[0]["driver_id"]
    loser  = group.iloc[1]["driver_id"]
    margin = group.iloc[1]["finish_position"] - group.iloc[0]["finish_position"]
    return pd.DataFrame({
        "season":  [group.iloc[0]["season"]],
        "round":   [group.iloc[0]["round"]],
        "team_id": [group.iloc[0]["team_id"]],
        "winner":  [winner],
        "loser":   [loser],
        "margin":  [margin],
    })

# group_keys=False prevents pandas >=1.5 from adding group keys to the index
results = (
    df.groupby(["season","round","team_id"], group_keys=False)
      .apply(teammate_result)
      .dropna()
      .reset_index(drop=True)
)

wins   = results.groupby("winner").size().reset_index(name="teammate_wins")
losses = results.groupby("loser").size().reset_index(name="teammate_losses")

driver_h2h = pd.merge(wins, losses, left_on="winner", right_on="loser", how="outer")
driver_h2h = driver_h2h.rename(columns={"winner":"driver_id"}).fillna(0)
driver_h2h["teammate_win_pct"] = (
    driver_h2h["teammate_wins"] / (driver_h2h["teammate_wins"] + driver_h2h["teammate_losses"])
)

df = df.merge(
    driver_h2h[["driver_id","teammate_wins","teammate_losses","teammate_win_pct"]],
    on="driver_id", how="left"
)
print(f"teammate_win_pct nulls: {df['teammate_win_pct'].isna().sum()}")
df.head()

teammate_win_pct nulls: 2


/var/folders/x3/jnkwnhwn35nd6y07yw7qm0380000gn/T/ipykernel_49902/1694900951.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(teammate_result)


,season,round,circuit_id,driver_id,driver_full_name,team_id,grid_position,finish_position,classified_status,points,...,var_delta_i_norm,avg_team_finish_position,rolling_avg_finish_position,career_starts_at_circuit,avg_finish_at_circuit,finish_grid_delta,avg_finish_grid_delta,teammate_wins,teammate_losses,teammate_win_pct
0,2018,1,Australian Grand Prix,VET,Sebastian Vettel,Ferrari,3.0,1.0,Finished,25.0,...,0.051316,2.0,2.0,0,NaN,-2.0,NaN,48.0,53.0,0.475248
1,2018,1,Australian Grand Prix,HAM,Lewis Hamilton,Mercedes,1.0,2.0,Finished,18.0,...,0.051316,5.0,5.0,0,NaN,1.0,NaN,104.0,73.0,0.587571
2,2018,1,Australian Grand Prix,RAI,Kimi Räikkönen,Ferrari,2.0,3.0,Finished,15.0,...,0.051316,2.0,2.0,0,NaN,1.0,NaN,48.0,31.0,0.607595
3,2018,1,Australian Grand Prix,RIC,Daniel Ricciardo,Red Bull Racing,8.0,4.0,Finished,12.0,...,0.051316,5.0,5.0,0,NaN,-4.0,NaN,54.0,74.0,0.421875
4,2018,1,Australian Grand Prix,ALO,Fernando Alonso,McLaren,10.0,5.0,Finished,10.0,...,0.051316,7.0,7.0,0,NaN,-5.0,NaN,87.0,51.0,0.630435


# 7. Quali Gap to Pole

`pole_q3_time` (fastest Q3 that weekend) and `quali_gap_to_pole` in seconds.

Drivers eliminated in Q1/Q2 will be NaN — expected, not a bug.

In [53]:
# best available quali time per driver (Q3 > Q2 > Q1, whichever exists)
df["best_quali_time"] = df["q3_time"].fillna(df["q2_time"]).fillna(df["q1_time"])

# pole time = fastest best_quali_time that weekend (across all drivers)
pole_times = (
    df.dropna(subset=["best_quali_time"])
      .groupby(["season","round"])["best_quali_time"]
      .min()
      .rename("pole_quali_time")
)
df = df.join(pole_times, on=["season","round"])

# gap to pole in seconds — now every driver who ran any quali session has a value
df["quali_gap_to_pole"] = (
    df["best_quali_time"] - df["pole_quali_time"]
).dt.total_seconds()

print(f"quali_gap_to_pole nulls: {df['quali_gap_to_pole'].isna().sum()} (should be ~222, only true DNS)")
df[["driver_id","q1_time","q2_time","q3_time","best_quali_time","quali_gap_to_pole"]].head(10)

quali_gap_to_pole nulls: 245 (should be ~222, only true DNS)


,driver_id,q1_time,q2_time,q3_time,best_quali_time,quali_gap_to_pole
0,VET,0 days 00:01:23.348000,0 days 00:01:21.944000,0 days 00:01:21.838000,0 days 00:01:21.838000,0.674
1,HAM,0 days 00:01:22.824000,0 days 00:01:22.051000,0 days 00:01:21.164000,0 days 00:01:21.164000,0.000
2,RAI,0 days 00:01:23.096000,0 days 00:01:22.507000,0 days 00:01:21.828000,0 days 00:01:21.828000,0.664
3,RIC,0 days 00:01:23.494000,0 days 00:01:22.897000,0 days 00:01:22.152000,0 days 00:01:22.152000,0.988
4,ALO,0 days 00:01:23.597000,0 days 00:01:23.692000,NaT,0 days 00:01:23.692000,2.528
5,VER,0 days 00:01:23.483000,0 days 00:01:22.416000,0 days 00:01:21.879000,0 days 00:01:21.879000,0.715
6,HUL,0 days 00:01:23.782000,0 days 00:01:23.544000,0 days 00:01:23.532000,0 days 00:01:23.532000,2.368
7,BOT,0 days 00:01:23.686000,0 days 00:01:22.089000,NaT,0 days 00:01:22.089000,0.925
8,VAN,0 days 00:01:24.073000,0 days 00:01:23.853000,NaT,0 days 00:01:23.853000,2.689
9,SAI,0 days 00:01:23.529000,0 days 00:01:23.061000,0 days 00:01:23.577000,0 days 00:01:23.577000,2.413


# 2. Team Pace Delta (quali-based, full 2018-2025)

Uses `quali_gap_to_pole` as the pace proxy — covers all seasons unlike the lap-time approach.

- `team_avg_pace_delta`: team's average quali gap to pole that race
- `rolling_avg_pace_delta`: rolling 3-race average (car development proxy)
- `pace_trend`: slope of pace over last 3 races (improving/declining)

In [54]:
def rolling_slope(x):
    y = x.values
    if len(y) < 2:
        return np.nan
    return np.polyfit(np.arange(len(y)), y, 1)[0]

team_pace = (
    df.groupby(["season","round","team_id"])["quali_gap_to_pole"]
      .mean()
      .reset_index(name="team_avg_pace_delta")
      .sort_values(["team_id","season","round"])
)
team_pace["rolling_avg_pace_delta"] = (
    team_pace.groupby("team_id")["team_avg_pace_delta"]
             .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)
team_pace["pace_trend"] = (
    team_pace.groupby("team_id")["team_avg_pace_delta"]
             .transform(lambda x: x.rolling(window=3, min_periods=2).apply(rolling_slope, raw=False))
)

df = df.merge(
    team_pace[["season","round","team_id","team_avg_pace_delta","rolling_avg_pace_delta","pace_trend"]],
    on=["season","round","team_id"], how="left"
)
print(f"team_avg_pace_delta nulls:    {df['team_avg_pace_delta'].isna().sum()}")
print(f"rolling_avg_pace_delta nulls: {df['rolling_avg_pace_delta'].isna().sum()}")
print(f"pace_trend nulls:             {df['pace_trend'].isna().sum()} (expected ~40-60, needs min 2 periods)")
df.head()

team_avg_pace_delta nulls:    201
rolling_avg_pace_delta nulls: 44
pace_trend nulls:             445 (expected ~40-60, needs min 2 periods)


,season,round,circuit_id,driver_id,driver_full_name,team_id,grid_position,finish_position,classified_status,points,...,avg_finish_grid_delta,teammate_wins,teammate_losses,teammate_win_pct,best_quali_time,pole_quali_time,quali_gap_to_pole,team_avg_pace_delta,rolling_avg_pace_delta,pace_trend
0,2018,1,Australian Grand Prix,VET,Sebastian Vettel,Ferrari,3.0,1.0,Finished,25.0,...,NaN,48.0,53.0,0.475248,0 days 00:01:21.838000,0 days 00:01:21.164000,0.674,0.6690,0.6690,NaN
1,2018,1,Australian Grand Prix,HAM,Lewis Hamilton,Mercedes,1.0,2.0,Finished,18.0,...,NaN,104.0,73.0,0.587571,0 days 00:01:21.164000,0 days 00:01:21.164000,0.000,0.4625,0.4625,NaN
2,2018,1,Australian Grand Prix,RAI,Kimi Räikkönen,Ferrari,2.0,3.0,Finished,15.0,...,NaN,48.0,31.0,0.607595,0 days 00:01:21.828000,0 days 00:01:21.164000,0.664,0.6690,0.6690,NaN
3,2018,1,Australian Grand Prix,RIC,Daniel Ricciardo,Red Bull Racing,8.0,4.0,Finished,12.0,...,NaN,54.0,74.0,0.421875,0 days 00:01:22.152000,0 days 00:01:21.164000,0.988,0.8515,0.8515,NaN
4,2018,1,Australian Grand Prix,ALO,Fernando Alonso,McLaren,10.0,5.0,Finished,10.0,...,NaN,87.0,51.0,0.630435,0 days 00:01:23.692000,0 days 00:01:21.164000,2.528,2.6085,2.6085,NaN


# 8. Circuit Wet Race Frequency

Fraction of historical races at this circuit that had rain.

In [55]:
wet_freq = (
    df.drop_duplicates(subset=["season","round"])
      .groupby("circuit_id")["race_had_rain"]
      .mean()
      .rename("circuit_wet_race_frequency")
)
df = df.join(wet_freq, on="circuit_id")
print(f"circuit_wet_race_frequency nulls: {df['circuit_wet_race_frequency'].isna().sum()}")
df[["circuit_id","race_had_rain","circuit_wet_race_frequency"]].drop_duplicates("circuit_id").head(10)

circuit_wet_race_frequency nulls: 0


,circuit_id,race_had_rain,circuit_wet_race_frequency
0,Australian Grand Prix,True,0.285714
20,Bahrain Grand Prix,False,0.000000
40,Chinese Grand Prix,True,0.200000
60,Azerbaijan Grand Prix,False,0.000000
80,Spanish Grand Prix,True,0.250000
100,Monaco Grand Prix,True,0.500000
120,Canadian Grand Prix,False,0.285714
140,French Grand Prix,True,0.250000
160,Austrian Grand Prix,False,0.375000
180,British Grand Prix,False,0.375000


# 9. Grid Penalties (Jolpica API)

`grid_penalty_places` and `had_grid_penalty`.

First run takes ~5 min. Re-running is instant (cached to `data/penalty_cache.json`).

In [56]:
BASE_URL    = "https://api.jolpi.ca/ergast/f1"
CACHE_FILE  = Path("./data/penalty_cache.json")
SLEEP_BASE  = 1.5
MAX_RETRIES = 5

def fetch_with_retry(url):
    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.get(url, timeout=15)
            if resp.status_code == 429:
                wait = 2 ** attempt * 3
                print(f"  [429] waiting {wait}s...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            print(f"  [ERROR] {e}")
            return None
    return None

def get_penalty_data(year, round_num):
    q = fetch_with_retry(f"{BASE_URL}/{year}/{round_num}/qualifying.json")
    r = fetch_with_retry(f"{BASE_URL}/{year}/{round_num}/results.json")
    if not q or not r:
        return None
    q_races = q["MRData"]["RaceTable"]["Races"]
    r_races = r["MRData"]["RaceTable"]["Races"]
    if not q_races or not r_races:
        return None
    quali_pos = {x["Driver"]["code"]: int(x["position"]) for x in q_races[0].get("QualifyingResults", [])}
    grid_pos  = {}
    for x in r_races[0].get("Results", []):
        g = x.get("grid", "0")
        grid_pos[x["Driver"]["code"]] = int(g) if g != "0" else None
    out = {}
    for code in set(quali_pos) | set(grid_pos):
        q_p, g_p = quali_pos.get(code), grid_pos.get(code)
        out[code] = max(0, g_p - q_p) if (q_p and g_p) else 0
    return out

print("Functions defined.")

Functions defined.


In [57]:
# fetch loop — safe to re-run, skips cached races
cache = json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}
races = df[["season","round"]].drop_duplicates().sort_values(["season","round"])
print(f"{len(races)} races total, {len(cache)} already cached")

for _, row in races.iterrows():
    year, rnd = int(row["season"]), int(row["round"])
    key = f"{year}_{rnd}"
    if key in cache:
        continue
    print(f"  fetching {year} R{rnd}...")
    result = get_penalty_data(year, rnd)
    if result is not None:
        cache[key] = result
        CACHE_FILE.write_text(json.dumps(cache))
    time.sleep(SLEEP_BASE)

print(f"Done — {len(cache)} races in cache")

180 races total, 173 already cached
  fetching 2026 R1...
  fetching 2026 R2...
  fetching 2026 R3...
  fetching 2026 R4...
  fetching 2026 R5...
  fetching 2026 R6...
  fetching 2026 R7...
Done — 180 races in cache


In [58]:
# map onto df
def lookup_penalty(row):
    key = f"{int(row['season'])}_{int(row['round'])}"
    val = cache.get(key, {}).get(row["driver_id"], 0)
    if isinstance(val, dict):          # handles old cache format
        return val.get("grid_penalty_places", 0)
    return val

df["grid_penalty_places"] = df.apply(lookup_penalty, axis=1)
df["had_grid_penalty"]    = df["grid_penalty_places"] > 0
print(f"Penalised rows: {df['had_grid_penalty'].sum()}")
df[df["had_grid_penalty"]][["season","round","circuit_id","driver_id","grid_penalty_places"]].head(10)

Penalised rows: 239


,season,round,circuit_id,driver_id,grid_penalty_places
3,2018,1,Australian Grand Prix,RIC,3
7,2018,1,Australian Grand Prix,BOT,5
22,2018,2,Bahrain Grand Prix,HAM,5
77,2018,4,Azerbaijan Grand Prix,HUL,5
93,2018,5,Spanish Grand Prix,SIR,1
114,2018,6,Monaco Grand Prix,GRO,3
153,2018,8,French Grand Prix,HAR,3
162,2018,9,Austrian Grand Prix,VET,3
168,2018,9,Austrian Grand Prix,LEC,5
219,2018,11,German Grand Prix,RIC,5


# 10. Weather Forecast (prediction-time only)

This function is NOT called during training. Call it at prediction time (race week) to get live forecast.

For historical training data, `race_had_rain` (already in the base CSV) and `circuit_wet_race_frequency` (Feature 8) are used instead.

In [59]:
from datetime import datetime

CIRCUIT_COORDS = {
    "Australian Grand Prix":    {"lat": -37.8497, "lon": 144.9680},
    "Bahrain Grand Prix":       {"lat": 26.0325,  "lon": 50.5106},
    "Saudi Arabian Grand Prix": {"lat": 21.6319,  "lon": 39.1044},
    "Japanese Grand Prix":      {"lat": 34.8431,  "lon": 136.5407},
    "Chinese Grand Prix":       {"lat": 31.3389,  "lon": 121.2198},
    "Miami Grand Prix":         {"lat": 25.9581,  "lon": -80.2389},
    "Emilia Romagna Grand Prix":{"lat": 44.3439,  "lon": 11.7167},
    "Monaco Grand Prix":        {"lat": 43.7384,  "lon": 7.4246},
    "Canadian Grand Prix":      {"lat": 45.5017,  "lon": -73.5220},
    "Spanish Grand Prix":       {"lat": 41.5700,  "lon": 2.2611},
    "Austrian Grand Prix":      {"lat": 47.2197,  "lon": 14.7647},
    "British Grand Prix":       {"lat": 52.0786,  "lon": -1.0169},
    "Hungarian Grand Prix":     {"lat": 47.5789,  "lon": 19.2486},
    "Belgian Grand Prix":       {"lat": 50.4372,  "lon": 5.9714},
    "Dutch Grand Prix":         {"lat": 52.3888,  "lon": 4.5409},
    "Italian Grand Prix":       {"lat": 45.6156,  "lon": 9.2811},
    "Azerbaijan Grand Prix":    {"lat": 40.3725,  "lon": 49.8533},
    "Singapore Grand Prix":     {"lat": 1.2899,   "lon": 103.8565},
    "United States Grand Prix": {"lat": 30.1328,  "lon": -97.6411},
    "Mexico City Grand Prix":   {"lat": 19.4042,  "lon": -99.0907},
    "São Paulo Grand Prix":     {"lat": -23.7036, "lon": -46.6997},
    "Las Vegas Grand Prix":     {"lat": 36.1699,  "lon": -115.1398},
    "Qatar Grand Prix":         {"lat": 25.4900,  "lon": 51.4542},
    "Abu Dhabi Grand Prix":     {"lat": 24.4672,  "lon": 54.6031},
}

OWM_API_KEY = "YOUR_API_KEY"  # replace with your free OpenWeatherMap key

def get_race_weather_forecast(circuit_id: str, race_date: str) -> dict:
    """
    PREDICTION TIME ONLY — not called during training.
    circuit_id: matches circuit_id column values exactly
    race_date:  "YYYY-MM-DD"
    Returns: {"forecast_rain_prob": float 0-1, "forecast_temp_c": float}
    """
    coords = CIRCUIT_COORDS.get(circuit_id)
    if not coords:
        print(f"[WARN] No coords for: {circuit_id}")
        return {"forecast_rain_prob": None, "forecast_temp_c": None}

    url = (
        f"https://api.openweathermap.org/data/2.5/forecast"
        f"?lat={coords['lat']}&lon={coords['lon']}"
        f"&appid={OWM_API_KEY}&units=metric"
    )
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    data = resp.json()

    forecasts = data.get("list", [])
    if not forecasts:
        return {"forecast_rain_prob": None, "forecast_temp_c": None}

    race_dt = datetime.strptime(race_date, "%Y-%m-%d").replace(hour=12)
    closest = min(
        forecasts,
        key=lambda f: abs(datetime.strptime(f["dt_txt"], "%Y-%m-%d %H:%M:%S") - race_dt)
    )
    return {
        "forecast_rain_prob": closest.get("pop", 0.0),
        "forecast_temp_c":    closest["main"]["temp"],
    }

print("get_race_weather_forecast() defined — call this at prediction time, not during training.")

get_race_weather_forecast() defined — call this at prediction time, not during training.


# Final: Null Check + Save

Run this cell last to verify and export `data/features.csv`.

In [60]:
print("── Shape:", df.shape)
print("\n── Null counts:")
print(df.isnull().sum())
print("\n── Dtypes:")
print(df.dtypes)

── Shape: (3612, 36)

── Null counts:
season                            0
round                             0
circuit_id                        0
driver_id                         0
driver_full_name                  0
team_id                           0
grid_position                    47
finish_position                  47
classified_status                44
points                           44
q1_time                         248
q2_time                        1100
q3_time                        1955
race_avg_air_temp                 0
race_avg_track_temp               0
race_had_rain                     0
delta_i                          47
var_delta_i_norm                 44
avg_team_finish_position         44
rolling_avg_finish_position       0
career_starts_at_circuit          0
avg_finish_at_circuit          1202
finish_grid_delta                47
avg_finish_grid_delta            44
teammate_wins                     2
teammate_losses                   2
teammate_win_pct          

In [61]:
df.to_csv("data/features.csv", index=False)
print(f"Saved data/features.csv — {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Saved data/features.csv — (3612, 36)
Columns: ['season', 'round', 'circuit_id', 'driver_id', 'driver_full_name', 'team_id', 'grid_position', 'finish_position', 'classified_status', 'points', 'q1_time', 'q2_time', 'q3_time', 'race_avg_air_temp', 'race_avg_track_temp', 'race_had_rain', 'delta_i', 'var_delta_i_norm', 'avg_team_finish_position', 'rolling_avg_finish_position', 'career_starts_at_circuit', 'avg_finish_at_circuit', 'finish_grid_delta', 'avg_finish_grid_delta', 'teammate_wins', 'teammate_losses', 'teammate_win_pct', 'best_quali_time', 'pole_quali_time', 'quali_gap_to_pole', 'team_avg_pace_delta', 'rolling_avg_pace_delta', 'pace_trend', 'circuit_wet_race_frequency', 'grid_penalty_places', 'had_grid_penalty']


In [62]:
print(df.columns.tolist())
print(df["team_avg_pace_delta"].describe())

['season', 'round', 'circuit_id', 'driver_id', 'driver_full_name', 'team_id', 'grid_position', 'finish_position', 'classified_status', 'points', 'q1_time', 'q2_time', 'q3_time', 'race_avg_air_temp', 'race_avg_track_temp', 'race_had_rain', 'delta_i', 'var_delta_i_norm', 'avg_team_finish_position', 'rolling_avg_finish_position', 'career_starts_at_circuit', 'avg_finish_at_circuit', 'finish_grid_delta', 'avg_finish_grid_delta', 'teammate_wins', 'teammate_losses', 'teammate_win_pct', 'best_quali_time', 'pole_quali_time', 'quali_gap_to_pole', 'team_avg_pace_delta', 'rolling_avg_pace_delta', 'pace_trend', 'circuit_wet_race_frequency', 'grid_penalty_places', 'had_grid_penalty']
count    3411.000000
mean        1.893680
std         2.499066
min         0.000000
25%         0.737000
50%         1.385000
75%         2.174000
max        31.148500
Name: team_avg_pace_delta, dtype: float64


In [63]:
print(df.groupby("season")["round"].nunique())

season
2018    21
2019    21
2020    17
2021    22
2022    22
2023    22
2024    24
2025    24
2026     7
Name: round, dtype: int64


In [64]:
print(df[["team_avg_pace_delta","rolling_avg_pace_delta","pace_trend","quali_gap_to_pole"]].isnull().sum())
print(f"\nTotal rows: {len(df)}")
print(f"\nSeasons covered:")
print(df.groupby("season")["round"].nunique())

team_avg_pace_delta       201
rolling_avg_pace_delta     44
pace_trend                445
quali_gap_to_pole         245
dtype: int64

Total rows: 3612

Seasons covered:
season
2018    21
2019    21
2020    17
2021    22
2022    22
2023    22
2024    24
2025    24
2026     7
Name: round, dtype: int64


In [65]:
df.to_csv("data/features.csv", index=False)
print(f"Saved — {df.shape}")

Saved — (3612, 36)
